In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df_int = pd.read_excel(r'C:\Users\hkhakan\OneDrive - Clemson University\Documents\GitHub\multi_period_disaster_logistics\posteriors (CK equations)\Data.xlsx', sheet_name="intensity", header=None, index_col=None) 
p_int = df_int.values
print("p_int: \n", p_int)

df_locx = pd.read_excel(r'C:\Users\hkhakan\OneDrive - Clemson University\Documents\GitHub\multi_period_disaster_logistics\posteriors (CK equations)\Data.xlsx', sheet_name="locx", header=None, index_col=None) 
p_locx = df_locx.values
print("p_locx: \n", p_locx)

df_locy = pd.read_excel(r'C:\Users\hkhakan\OneDrive - Clemson University\Documents\GitHub\multi_period_disaster_logistics\posteriors (CK equations)\Data.xlsx', sheet_name="locy-det", header=None, index_col=None) 
p_locy = df_locy.values
print("p_locy: \n", p_locy)

In [ ]:
#np.set_printoptions(threshold=np.inf)

k = np.kron(p_int, p_locx)
TM = np.kron(k, p_locy)
print ("Transition probability matrix:\n", TM)

no_errors = True

for i in range(TM.shape[0]):
    row_sum = np.sum(TM[i, :])
    if abs(row_sum - 1.0) > 1e-6:
        print(f"Error: Row {i + 1} has a sum of {row_sum}")
        no_errors = False
        
if no_errors:
    print("All rows are valid.")

In [ ]:
intensity_states = [0, 1, 2, 3, 4, 5]
x_coordinates = [50, 150, 250, 350, 450, 550, 650]
y_coordinates = [-600, -500, -400, -300, -200, -100, 0]

states = []
for intensity in intensity_states:
    for x in x_coordinates:
        for y in y_coordinates:
            states.append((intensity, x, y))

for idx, state in enumerate(states):
    print(f"State {idx + 1}: {state}")

def get_state_idx(init_state, states):
    return states.index(init_state)

def init_prior(init_state, states, num_states=294):
    P0 = np.zeros((1, num_states))
    state_idx = get_state_idx(init_state, states)
    P0[0, state_idx] = 1
    return P0

init_state = (1, 150, -600)

P0 = init_prior(init_state, states)

print(f"\nThe first prior distribution (P0) for initial state {init_state}:\n", P0)


In [ ]:
# Define a function to check if the absorbing state is reached
def check_abso(P_next, states):
    for i, prob in enumerate(P_next.flatten()):
        if prob > 0:  # Check for positive probability
            intensity, x, y = states[i]  # Unpack the state (intensity, x-coordinate, y-coordinate)
            if y != 0:  # If any state with positive probability has y != 0, absorbing state is not reached
                return False
    return True  # All positive probabilities correspond to y = 0 (absorbing state)

# Initialize variables
posteriors = []
P_previous = P0

t = 1  # Start with the first time step
abso_reached = False  # Set the absorbing state flag to False

# Loop until we reach the absorbing state
while not abso_reached:
    # Calculate the next posterior distribution
    P_next = P_previous @ TM
    
    print(f"\nPosterior distribution at t={t}: \n", P_next)
    
    # Sum the rows of P_next for validation (optional)
    row_sums = np.sum(P_next, axis=1)
    print(f"Sum of each row of P{t}:\n", row_sums)

    # Append the posterior to the list
    posteriors.append(P_next.flatten())
    
    # Check if the absorbing state is reached
    abso_reached = check_abso(P_next, states)

    # Update for the next step
    P_previous = P_next
    t += 1  # Increment the time step

# Create the DataFrame and save to Excel
df_posteriors = pd.DataFrame(posteriors, index=[f"t={t}" for t in range(1, t)])
output_file = r'C:\Users\hkhakan\OneDrive - Clemson University\Documents\GitHub\multi_period_disaster_logistics\posteriors (CK equations)\pos_dist_det.xlsx'
df_posteriors.to_excel(output_file, sheet_name='Posteriors')

print(f"Posterior distributions have been saved to {output_file}")

In [ ]:
#How to validate the result 1

TM_final = np.linalg.matrix_power(TM, 6)

pos_final = P0 @ TM_final

equal = np.allclose(pos_final, P_next)
if equal:
    print("\nThe matrices are equal!")
else:
    print("\nThe matrices are NOT equal!")

df_posteriors.loc['t=final'] = pos_final.flatten()

output_file = r'C:\Users\hkhakan\OneDrive - Clemson University\Documents\GitHub\multi_period_disaster_logistics\posteriors (CK equations)\pos_dist_det.xlsx'
df_posteriors.to_excel(output_file, sheet_name='Posteriors')

print(f"Posterior distributions and pos_final have been saved to {output_file}")

In [ ]:
#How to validate the result 2

n_x = len(x_coordinates)
n_y = len(y_coordinates)
n_intensity = len(intensity_states)

reshaped_posteriors = [np.reshape(posterior, (n_x, n_y, n_intensity)) for posterior in posteriors]

rows_list = []

for t_step, posterior_3d in enumerate(reshaped_posteriors, start=1):
    for i, x in enumerate(x_coordinates):
        for j, y in enumerate(y_coordinates):
            for k, intensity in enumerate(intensity_states):
                rows_list.append({
                    'Time': f't={t_step}',
                    'x': x,
                    'y': y,
                    'intensity': intensity,
                    'posterior': posterior_3d[i, j, k]
                })

df_posteriors = pd.DataFrame(rows_list)

output_file = r'C:\Users\hkhakan\OneDrive - Clemson University\Documents\GitHub\multi_period_disaster_logistics\posteriors (CK equations)\reshaped_pos_dist_det.xlsx'
df_posteriors.to_excel(output_file, sheet_name='Posteriors', index=False)

print(f"Reshaped posterior distributions have been saved to {output_file}")
